# OpenGrid — GRPO Training Notebook (Unsloth variant)

**End-to-end GRPO fine-tuning of Qwen2.5-1.5B on OpenGrid, accelerated by [Unsloth](https://unsloth.ai/).**

This notebook is the Unsloth-equivalent of `opengrid_grpo_colab.ipynb`. The pipeline (env-grounded reward, baseline + post-training eval, plots, summary.json) is identical — only the model loading + training kernel are swapped.

**Why Unsloth?** ~2× faster training, lower VRAM at the same LoRA config. Same scientific outcome.

**Why two notebooks?** The shipped run used the standard `transformers + bitsandbytes + peft` stack (matches `run_training.py`). This Unsloth notebook is provided as an alternative path — useful if you want to retrain faster or fit on a smaller GPU.

**Hardware:** Designed for Colab T4 (free) or A10G/L4 (paid). Will not work on CPU.


## 1. Install Dependencies (Unsloth)


In [ ]:
%%capture
# Unsloth pins compatible versions of transformers/trl/peft itself.
# This single command installs everything needed.
!pip install -q unsloth unsloth_zoo
!pip install -q --no-deps trl==0.15.2 peft accelerate bitsandbytes
!pip install -q xformers triton
!pip install -q datasets fastapi uvicorn pydantic numpy networkx matplotlib httpx


## 2. Clone OpenGrid Repository

In [ ]:
import os

#  UPDATE THIS with your actual repo URL
REPO_URL = "https://github.com/krishnagoyal099/Opengrid_env.git"

if not os.path.exists("opengrid"):
    !git clone {REPO_URL} opengrid
else:
    !cd opengrid && git pull

os.chdir("opengrid")
print(f"Working directory: {os.getcwd()}")
!ls -la

## 3. Verify GPU & Environment

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print(" No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# Verify OpenGrid imports work
import sys
sys.path.insert(0, '.')

from src.environment import OpenGridEnv
from src.tasks import TASKS
from src.models import GridAction, BusAdjustment

print(f"Available tasks: {list(TASKS.keys())}")
for tid, cfg in TASKS.items():
    print(f"  {tid}: {cfg['num_buses']} buses, {cfg['num_agents']} agents, {cfg.get('difficulty','')}")

## 4. Run Test Mode (Pipeline Verification)

In [ ]:
!python training/train_grpo.py --test-mode

## 5. Baseline Evaluation (Before Training)

Run the heuristic policy to get baseline scores. We'll compare against this after training.

In [ ]:
import os, json, re, copy, pickle
import numpy as np
from src.environment import OpenGridEnv
from src.tasks import TASKS
from src.models import GridAction, BusAdjustment
from training.train_grpo import (
    rollout_multi_agent, format_observation_prompt, extract_action
)

def heuristic_generate(prompt):
    """Simple proportional controller — same baseline as run_training.py."""
    freq_match = re.search(r'Frequency: ([\d.]+)', prompt)
    freq = float(freq_match.group(1)) if freq_match else 50.0
    error = 50.0 - freq
    delta = max(-20, min(20, error * 10))
    bus_match = re.search(r'Bus (\d+) \((generator|battery|slack)\)', prompt)
    if bus_match:
        return json.dumps({"bus_adjustments": [{"bus_id": int(bus_match.group(1)), "delta": round(delta, 1)}], "topology_actions": []})
    return json.dumps({"bus_adjustments": [], "topology_actions": []})

# Evaluate baseline on all 6 tasks × 3 episodes (matches run_training.py)
BASELINE_TASKS = [
    "task_easy", "task_medium",
    "karnataka_easy", "karnataka_medium", "karnataka_hard",
    "task_karnataka",
]
baseline_results = {}
for task_id in BASELINE_TASKS:
    if task_id not in TASKS:
        continue
    config = TASKS[task_id]
    rewards = []
    for ep in range(3):
        ep_config = copy.deepcopy(config)
        ep_config['seed'] = 42 + ep
        env = OpenGridEnv(ep_config)
        result = rollout_multi_agent(env, heuristic_generate, ep_config)
        rewards.append(result['total_reward'])
    baseline_results[task_id] = {
        "avg":     float(np.mean(rewards)),
        "std":     float(np.std(rewards)),
        "rewards": rewards,
    }
    print(f"  [BASELINE] {task_id:<20} {np.mean(rewards):>8.2f} ± {np.std(rewards):.2f}")

os.makedirs("training/outputs", exist_ok=True)
with open("training/outputs/baseline_results.pkl", "wb") as f:
    pickle.dump(baseline_results, f)
print(f"\nBaseline saved ({len(baseline_results)} tasks).")

## 6. Load Model — `Qwen2.5-1.5B-Instruct` via Unsloth FastLanguageModel

Unsloth handles 4-bit quantization, LoRA, and gradient checkpointing in one call. We use the pre-quantized `unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit` for fast loading.


In [ ]:
# IMPORTANT: import unsloth BEFORE transformers so its patches apply.
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch

MODEL_NAME  = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"
LORA_RANK   = 16
MAX_SEQ_LEN = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,            # auto: bf16 if supported, else fp16
    load_in_4bit=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_RANK * 2,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",  # Unsloth's optimized checkpoint kernel
    random_state=42,
    use_rslora=False,
)
model.config.pad_token_id = tokenizer.pad_token_id

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Model:  {MODEL_NAME}")
print(f"Trainable params: {trainable:,} ({100 * trainable / total:.2f}% of {total:,})")
print(f"BF16 supported: {is_bfloat16_supported()}")


## 7. Generate Training Prompts from Environment

In [ ]:
import copy
import json as _json
import numpy as np
from training.train_grpo import SYSTEM_PROMPT, format_observation_prompt

# ── Iteration budget ─────────────────────────────────────────
# A10G run (full):  NUM_EPISODES = 10  →  ~600 prompts, 3 epochs ≈ 160 min
# T4 smoke test:    NUM_EPISODES = 8   →  ~480 prompts, 1 epoch  ≈ 45 min
TRAIN_TASK   = "task_karnataka"
NUM_EPISODES = 8
# ─────────────────────────────────────────────────────────────

task_config = copy.deepcopy(TASKS[TRAIN_TASK])
base_seed   = task_config.get('seed', 42)
rng         = np.random.RandomState(base_seed)

prompts = []
obs_contexts = []  # JSON-string scalars (Arrow-friendly)

for episode in range(NUM_EPISODES):
    ep_config = copy.deepcopy(task_config)
    ep_config['seed'] = base_seed + episode
    env = OpenGridEnv(ep_config)
    zone_obs = env.reset_multi()

    # Adversarial: drain batteries every 5th episode → forces the policy
    # to learn recovery, not just steady-state.
    if episode % 5 == 0:
        for b in env.bus_state:
            b_cfg = env._find_bus_config(b['id'])
            if b_cfg and b_cfg['type'] == 'battery':
                b['soc'] = max(1.0, b['soc'] * 0.1)

    for t in range(min(15, task_config['max_steps'])):
        for agent_id, obs in zone_obs.items():
            obs_dict = _json.loads(obs.model_dump_json())
            prompt_text = format_observation_prompt(obs_dict, zone_name=obs.zone_name)
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt_text},
            ]
            formatted = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            prompts.append(formatted)
            obs_contexts.append(_json.dumps(obs_dict))

        # Advance env with diverse random actions (1–3 controllable buses, ±30 delta)
        random_actions = {}
        for aid in range(env.num_agents):
            zone_buses = task_config['zone_bus_ids'].get(aid, [])
            controllable = [
                bid for bid in zone_buses
                if next((b for b in task_config['buses'] if b['id'] == bid), {}).get('type')
                in ['generator', 'battery']
            ]
            adj = []
            if controllable:
                n_adj  = min(len(controllable), rng.randint(1, 3))
                chosen = rng.choice(controllable, size=n_adj, replace=False)
                for bid in chosen:
                    adj.append(BusAdjustment(bus_id=int(bid),
                                             delta=float(rng.uniform(-30, 30))))
            random_actions[aid] = GridAction(bus_adjustments=adj)

        result = env.step_multi(random_actions)
        if result.done:
            break
        zone_obs = result.observations

print(f"Generated {len(prompts)} training prompts from {NUM_EPISODES} episodes")
print(f"\nSample prompt (first 400 chars):")
print(prompts[0][:400])

## 8. Define GRPO Reward Function

In [ ]:
import json as _json
from training.train_grpo import compute_grpo_reward_env, extract_action

def reward_fn(completions, obs_context=None, **kwargs):
    """GRPO reward function with env-grounded physics rewards."""
    texts = []
    for c in completions:
        if isinstance(c, list):
            text = c[-1]["content"] if c else ""
        else:
            text = str(c)
        texts.append(text)

    if obs_context is None:
        batch_obs = [None] * len(texts)
    else:
        batch_obs = [
            _json.loads(ctx) if isinstance(ctx, str) else ctx
            for ctx in obs_context
        ]
    return compute_grpo_reward_env(texts, batch_obs, task_config, horizon=3)

# Sanity test
test_rewards = reward_fn([
    '{"bus_adjustments": [{"bus_id": 1, "delta": 5.0}], "topology_actions": []}',
    "invalid json here",
])
print(f"Test rewards: {test_rewards}")
assert len(test_rewards) == 2
print("[OK] reward_fn works")


## 9. Train with GRPO

We use TRL's `GRPOTrainer` with the same hyperparameters as the shipped run. The only difference: `gradient_checkpointing=False` here because Unsloth's `use_gradient_checkpointing="unsloth"` already wires up its own (faster) checkpoint kernel.


In [ ]:
import time, inspect as _inspect
from trl import GRPOTrainer, GRPOConfig
from transformers import GenerationConfig
from datasets import Dataset

_cuda_ok = torch.cuda.is_available()
_bf16    = _cuda_ok and torch.cuda.is_bf16_supported()
_fp16    = _cuda_ok and not _bf16
_grpo_params = set(_inspect.signature(GRPOConfig.__init__).parameters)

# Pin generation config so EOS is always respected (avoids generations
# always running to max_completion_length).
model.generation_config = GenerationConfig(
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
    max_new_tokens=64,
)

# Iteration budget — full A10G run used NUM_EPOCHS=3, T4 use 1 to fit time.
NUM_EPOCHS = 1   # set to 3 to reproduce the full summary.json run
SAVE_STEPS = 25  # checkpoint often so a late crash still saves progress

# Some GRPOConfig params were renamed/moved between TRL versions;
# only pass what this installed TRL accepts.
_opt = {}
if 'max_prompt_length'     in _grpo_params: _opt['max_prompt_length']     = 512
if 'max_completion_length' in _grpo_params: _opt['max_completion_length'] = 64
if 'torch_compile'         in _grpo_params: _opt['torch_compile']         = False
if 'use_vllm'              in _grpo_params: _opt['use_vllm']              = False

grpo_config = GRPOConfig(
    output_dir="training/outputs/grpo_checkpoints_unsloth",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    logging_steps=1,
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    num_generations=4,
    report_to="none",
    remove_unused_columns=False,
    bf16=_bf16,
    fp16=_fp16,
    gradient_checkpointing=False,  # Unsloth handles this internally
    optim="paged_adamw_8bit",
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    dataloader_num_workers=0,
    **_opt,
)

train_dataset = Dataset.from_dict({"prompt": prompts, "obs_context": obs_contexts})
print(f"Dataset:        {len(train_dataset)} rows, columns: {train_dataset.column_names}")
print(f"Effective batch: {grpo_config.per_device_train_batch_size * grpo_config.gradient_accumulation_steps}")
print(f"Epochs:          {NUM_EPOCHS}")

FastLanguageModel.for_training(model)

trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    train_dataset=train_dataset,
    reward_funcs=reward_fn,
    processing_class=tokenizer,
)

# Sanity-check generation BEFORE handing off to GRPO.
# If this hangs, the model/tokenizer setup is the culprit, not GRPO.
print("\n[SANITY] Testing model.generate() (should finish in <30s)...")
_t0 = time.time()
_test_inputs = tokenizer("Hello", return_tensors="pt").to(model.device)
with torch.no_grad():
    _out = model.generate(
        **_test_inputs, max_new_tokens=8, do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
print(f"[SANITY] OK ({time.time()-_t0:.1f}s): {tokenizer.decode(_out[0][-8:], skip_special_tokens=True)!r}")

print("\n[NOTE] First GRPO step includes Triton JIT — may show 0/N for up to 5 min. That is normal.")
t0 = time.time()
train_result = trainer.train()
train_time = time.time() - t0

print(f"\nTraining complete in {train_time/60:.1f} minutes")
print(f"Total steps: {trainer.state.global_step}")

## 10. Save Trained Model (Unsloth)


In [ ]:
import torch
OUTPUT_PATH = "training/outputs/trained_model_unsloth"
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Save adapter only — avoids OOM from merging/dequantising the full 4-bit model.
# This is what run_training.py does on the A10G; matters even more on T4.
torch.cuda.empty_cache()
try:
    model.save_pretrained(OUTPUT_PATH)        # LoRA adapter weights only
    tokenizer.save_pretrained(OUTPUT_PATH)
    print(f"Adapter saved to {OUTPUT_PATH}")
except Exception as save_err:
    print(f"WARNING: adapter save failed ({save_err}); training metrics still captured")

## 11. Evaluate Trained Model (Unsloth Inference Mode)

Switch the model into Unsloth's inference fast-path before generation — gives ~2× faster decoding than the standard `transformers` path.


In [ ]:
FastLanguageModel.for_inference(model)

torch.cuda.empty_cache()
model.eval()

def trained_generate(prompt):
    """Generate action with the trained adapter — same as run_training.py."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs   = tokenizer(formatted, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,    # short for speed; enough for JSON action
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

# Same representative subset as run_training.py — keeps eval within VRAM budget
EVAL_TASKS = ["task_easy", "task_karnataka", "karnataka_hard"]
trained_results = {}

for task_id in EVAL_TASKS:
    if task_id not in TASKS:
        continue
    try:
        config    = TASKS[task_id]
        ep_config = copy.deepcopy(config)
        ep_config['seed'] = 42
        env    = OpenGridEnv(ep_config)
        result = rollout_multi_agent(env, trained_generate, ep_config)
        r      = result['total_reward']
        trained_results[task_id] = {"avg": float(r), "std": 0.0, "rewards": [r]}
        print(f"  [TRAINED] {task_id:<20} {r:>8.2f}   blackout={result['is_blackout']}")
        torch.cuda.empty_cache()
    except Exception as eval_err:
        print(f"  [TRAINED] {task_id:<20} eval failed ({eval_err})")
        trained_results[task_id] = {"avg": None, "std": None, "rewards": []}

## 12. Generate Plots & summary.json (Unsloth)


In [ ]:
import matplotlib.pyplot as plt
import pickle

# Re-load baseline (in case Cell 11 was run in a different session)
with open("training/outputs/baseline_results.pkl", "rb") as f:
    baseline_results = pickle.load(f)

# ── Plot 1: Before vs After bar chart ──
# Only include tasks where the trained eval succeeded (avg is not None)
common_tasks = [t for t in baseline_results
                if t in trained_results and trained_results[t]['avg'] is not None]

fig, ax = plt.subplots(figsize=(10, 6))
x     = np.arange(len(common_tasks))
width = 0.35

before_vals = [baseline_results[t]['avg'] for t in common_tasks]
after_vals  = [trained_results[t]['avg']  for t in common_tasks]

bars1 = ax.bar(x - width/2, before_vals, width, label='Heuristic Baseline', color='#ff6b6b', alpha=0.85)
bars2 = ax.bar(x + width/2, after_vals,  width, label='GRPO Trained',       color='#00d4aa', alpha=0.85)

ax.set_xlabel('Task', fontsize=12)
ax.set_ylabel('Average Episode Reward', fontsize=12)
ax.set_title('OpenGrid — GRPO Training: Before vs After', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([t.replace('task_', '').replace('karnataka_', 'KA-').title() for t in common_tasks],
                   rotation=15, ha='right')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(0, color='black', linewidth=0.6, alpha=0.4)

for bars in (bars1, bars2):
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2.,
                h + (1.5 if h >= 0 else -3),
                f'{h:.1f}',
                ha='center', va='bottom' if h >= 0 else 'top', fontsize=9)

plt.tight_layout()
plt.savefig('training/outputs/before_after_unsloth.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: training/outputs/before_after_unsloth.png")

In [ ]:
# ── Plots 2 & 3: Training loss + reward curves ──
history = trainer.state.log_history

# Loss
loss_steps  = [h['step'] for h in history if 'loss'   in h]
losses      = [h['loss'] for h in history if 'loss'   in h]
# Reward (GRPO logs `reward` per step — this is THE plot judges look for)
rew_steps   = [h['step']   for h in history if 'reward' in h]
rewards     = [h['reward'] for h in history if 'reward' in h]

if loss_steps:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(loss_steps, losses, color='#ff6b6b', linewidth=1.0, alpha=0.45, label='Loss')
    if len(losses) > 10:
        w = min(20, len(losses) // 3)
        smoothed = np.convolve(losses, np.ones(w)/w, mode='valid')
        ax.plot(loss_steps[w-1:], smoothed, color='#e03131', linewidth=2.5, label=f'Smoothed (w={w})')
    ax.set_xlabel('Training Step'); ax.set_ylabel('Loss')
    ax.set_title('OpenGrid GRPO — Training Loss', fontweight='bold')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('training/outputs/training_loss.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: training/outputs/training_loss.png")

if rew_steps:
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(rew_steps, rewards, color='#4dabf7', linewidth=0.8, alpha=0.5, label='Reward (per step)')
    if len(rewards) > 10:
        w = min(20, len(rewards) // 3)
        sm = np.convolve(rewards, np.ones(w)/w, mode='valid')
        ax.plot(rew_steps[w-1:], sm, color='#00d4aa', linewidth=2.5, label=f'Smoothed (w={w})')
    ax.axhline(0, color='#ff6b6b', linestyle='--', linewidth=1, alpha=0.7, label='Zero baseline')
    ax.set_xlabel('Training Step'); ax.set_ylabel('GRPO Reward')
    ax.set_title('OpenGrid GRPO — Reward Curve\n(Qwen2.5-1.5B-Instruct, LoRA r=16, task_karnataka)', fontweight='bold')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('training/outputs/training_reward_curve.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: training/outputs/training_reward_curve.png")

# ── summary.json (matches run_training.py format) ──
summary = {
    "model":          MODEL_NAME,
    "train_task":     TRAIN_TASK,
    "train_time_minutes": round(train_time / 60, 1),
    "num_prompts":    len(prompts),
    "num_epochs":     NUM_EPOCHS,
    "num_steps":      trainer.state.global_step,
    "lora_rank":      LORA_RANK,
    "framework":      "TRL GRPOTrainer + bitsandbytes 4-bit",
    "reward_start":   round(float(np.mean(rewards[:5])),  4) if rewards else None,
    "reward_end":     round(float(np.mean(rewards[-20:])),4) if rewards else None,
    "reward_peak":    round(float(max(rewards)),          4) if rewards else None,
    "baseline":       {k: {"avg": round(v["avg"], 2), "std": round(v["std"], 2)}
                       for k, v in baseline_results.items()},
    "trained":        {k: {"avg": round(v["avg"], 2) if v["avg"] is not None else None,
                           "std": round(v["std"], 2) if v["std"] is not None else None}
                       for k, v in trained_results.items()},
}
with open("training/outputs/summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("Saved: training/outputs/summary.json")

## 13. Summary & Next Steps

### Results Table

In [ ]:
print("="*60)
print("  OpenGrid GRPO Training — Results Summary")
print("="*60)

common_tasks = [t for t in baseline_results
                if t in trained_results and trained_results[t]['avg'] is not None]

print(f"{'Task':<20} {'Baseline':>12} {'Trained':>12} {'Δ':>10}")
print("-"*60)
for t in common_tasks:
    b = baseline_results[t]['avg']
    a = trained_results[t]['avg']
    delta = a - b
    arrow = '↑' if delta > 0 else '↓'
    print(f"{t:<20} {b:>10.2f}   {a:>10.2f}   {arrow} {abs(delta):.2f}")
print("="*60)
print(f"\nTraining time: {train_time/60:.1f} min  ·  Steps: {trainer.state.global_step}")
if rewards:
    print(f"GRPO reward:   {np.mean(rewards[:5]):.3f}  →  {np.mean(rewards[-20:]):.3f}  (peak {max(rewards):.3f})")

In [ ]:
# Display all generated plots + summary inline
from IPython.display import Image, display, JSON
import os, json

for img in ("training_reward_curve.png", "training_loss.png", "before_after.png"):
    p = f"training/outputs/{img}"
    if os.path.exists(p):
        display(Image(p))

if os.path.exists("training/outputs/summary.json"):
    with open("training/outputs/summary.json") as f:
        display(JSON(json.load(f)))
